# Section 1: Data Loading & Cleaning


### Import Neccessary Libraries

In [ ]:
import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from scipy import stats

# ====================== DISPLAY SETTINGS ======================
pd.options.display.float_format = "{:,.6f}".format   # 6 decimal places + thousand separators
pd.set_option('display.max_columns', None)           # Show all columns
pd.set_option('display.width', 1000)                 # Wider display
pd.set_option('display.max_rows', 100)               # Adjust as needed
# ============================================================

sns.set_style("whitegrid")   # Clean plotting style
sns.set()

### Data Loading, Cleaning and Conversion into CSV files

In [ ]:
# Dangote Cement Stock Price
dangcem = pd.read_csv("../data/raw/Dangote Cement Stock Price History.csv", index_col="Date") 
dangcem.drop(columns=["Open", "High", "Low", "Vol.", "Change %"], inplace=True)
dangcem.rename(columns={"Price":"DANGCEM"}, inplace=True)
# Covert str => float
dangcem["DANGCEM"] = dangcem["DANGCEM"].str.replace(",","").astype("float64")

# Guaranty Trust Holding Stock Price
gtco = pd.read_csv("../data/raw/Guaranty Trust Holding Stock Price History.csv", index_col="Date") 
gtco.drop(columns=["Open", "High", "Low", "Vol.", "Change %"], inplace=True)
gtco.rename(columns={"Price":"GTCO"}, inplace=True)

# Zenith Bank Stock Price
zenith = pd.read_csv("../data/raw/Zenith Bank Stock Price History.csv", index_col="Date") 
zenith.drop(columns=["Open", "High", "Low", "Vol.", "Change %"], inplace=True)
zenith.rename(columns={"Price":"ZENITHB"}, inplace=True)

In [ ]:
stock_prices = pd.concat([dangcem, gtco, zenith], axis=1)
stock_prices.to_csv("../data/processed/Dangcem_Gtco_Zenithb.csv", index=True)

### Loading the CSV file I created.

In [ ]:
start = pd.Timestamp("2021-06-21").tz_localize(None)
end = pd.Timestamp("2026-06-21").tz_localize(None)

prices = pd.read_csv("../data/processed/Dangcem_Gtco_Zenithb.csv", index_col="Date", parse_dates=True)

prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

prices = prices.loc[start:end]
# prices = prices[prices.index.dayofweek < 5]
prices.head()

# Section 2: Data Exploration & Quality Check

### Inspect data, handle missing values, plot prices.

In [ ]:
prices.dropna(inplace=True)
prices.head()

In [ ]:
prices.index.day_name().unique()

In [ ]:
prices.describe()

In [ ]:
prices.dtypes

In [ ]:
prices.isna().sum()

In [ ]:
duplicates = prices.index[prices.index.duplicated()]
print(len(duplicates), "duplicate dates found")

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=prices)
plt.xlabel("Year")
plt.ylabel("Stock Prices")
plt.title("DANGCEM, GTCO and ZENITHB Equities Performance")
plt.show()

# Section 3: Return Calculation & Analysis



### I) Computing returns for all three stocks.

In [ ]:
#SIMPLE RETURNS
# simple_returns = prices.pct_change().iloc[1:]

# LOG RETURNS
returns = np.log(prices) - np.log(prices.shift(1))
returns.dropna(inplace=True)

returns.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(returns)
plt.xlabel("Year")
plt.ylabel("Stock Returns")
plt.title("DANGCEM, GTCO and ZENITHB Equities (Simple) Returns")
plt.show()

### II) Annualized return and volatility.

#### Daily Return

In [ ]:
summary_df = pd.DataFrame()

In [ ]:
daily_return = returns.mean()
print("===========================")
print("Daily Average Return")
print("===========================")
print(daily_return)
print("===========================")
summary_df["DailyReturn"] = daily_return


In [ ]:
ax1 = returns.plot(figsize=(15, 3), y="GTCO", title="Figure 1: S&P 500 Daily Returns")
ax2 = returns.plot(figsize=(15, 3), y="ZENITHB", title="Figure 2: Zenith Bank Daily Returns")
ax3= returns.plot(figsize=(15, 3), y="DANGCEM", title="Figure 3: Dangote Cement Daily Returns")
# Set y-axis limits for both plots
ax1.set_ylim(-0.5, 0.4)
ax2.set_ylim(-0.5, 0.4)
ax3.set_ylim(-0.5, 0.4);


#### Annualized Return

In [ ]:
# Using Log returns`
N = 252 # Trading days

# Geometric Annualized Return (Compounded)
annualized_return = np.prod(1+returns, axis=0)**(1/N) - 1 # Using Geometric Mean
summary_df["AnnualizedReturn"] = annualized_return

#### Volatility (Standard Deviation)

In [ ]:
vols = returns.std()
summary_df["StandardDeviation"] = vols
print("===========================")
print("Volatility")
print("===========================")
print(vols)
print("===========================")

### III) Rolling volatility.

In [ ]:
#  Rolling Volatility for 20- day

rolling_20 = (abs(prices - prices.rolling(20).mean())/prices).iloc[19:, :]
# rolling_20

#  Rolling Volatility for 50- day

rolling_50 = (abs(prices - prices.rolling(50).mean())/prices).iloc[49:, :]
# rolling_50.head()

#### 4. Moving Average Volatility


In [ ]:
# 50-day Moving Average Volatility
moving_average_vol = rolling_20.mean()

# moving_average_vol = (abs(prices - prices.rolling(20).mean())/prices).mean()
print("===========================")
print("20-Day Moving Average")
print("===========================")

print(moving_average_vol)
print("==========================================")

# 50-day Moving Average Volatility
moving_average_vol = rolling_50.mean()
summary_df["MovingAverageVolatility"] = moving_average_vol
# moving_average_vol = (abs(prices - prices.rolling(50).mean())/prices).mean()
print("50-Day Moving Average")
print("===========================")
print(moving_average_vol)
print("===========================")


In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(x="Date", y="DANGCEM", data=rolling_50)
plt.xlabel("Year")
plt.ylabel("Rolling Volatility")
plt.title("Dangote Cement 50-Days Rolling Volatilty")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(x="Date", y="ZENITHB", data=rolling_50)
plt.xlabel("Year")
plt.ylabel("Rolling Volatility")
plt.title("Zenith Bank 50-Days Rolling Volatilty")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(x="Date", y="GTCO", data=rolling_50, legend=True)
plt.xlabel("Year")
plt.ylabel("Rolling Volatility")
plt.title("Guarantee Trust Bank 50-Day Rolling Volatilty")
plt.show()

#### 5. Sharpe Ratio

In [ ]:
# Assuming it is a zero riskfree rate
sharpe_ratio = returns.mean()/returns.std()
summary_df["SharpeRatio"] = sharpe_ratio
print("===============================")
print("Sharpe Ratio")
print("===============================")
print(sharpe_ratio)
print("===============================")

#### 6. Semi-variance (Deviation from Average Return)

In [ ]:
semivariance = ((returns[returns < returns.mean()] - returns.mean())**2).mean()
summary_df["SemiVariance"]= semivariance
print(semivariance)
dangcem_mean = returns["DANGCEM"].mean()
gtco_mean = returns["GTCO"].mean()
zenithb_mean = returns["ZENITHB"].mean()


This output shows that ZENITHB has a much higher semivariance (0.000474) compared to the GTCO (0.000414) and DANGCEM (0.000236). This indicates that ZENITHB has experienced significantly larger negative deviations from its average return, suggesting higher downside risk.

### IV) High-Low range analysis.

In [ ]:
start = pd.Timestamp(end - datetime.timedelta(365))
# start = end - pd.DateOffset(years=1)

In [ ]:
currYear = prices.loc[start:end]

In [ ]:
low = currYear.min()
high = currYear.max()
high_low = (high - low)/prices.iloc[-1, :]
summary_df["HighMinusLow"] = high_low
print("===============================")
print("High Minus Low Range Analysis")
print("===============================")
print(high_low)
print("===============================")

# Section 4: Matrix Analysis


### I) Covariance Matrix

In [ ]:
covariance = returns.cov()
covariance

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    covariance,
    annot=True,
    cmap="coolwarm",
    fmt=".6f",
    xticklabels=covariance.columns,
    yticklabels=covariance.columns
)
plt.title("Covariance Heatmap")
plt.show()


### II) Correlation Matrix

In [ ]:
corr = returns.corr()
corr

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".6f")
plt.title("Covariance Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
sns.regplot(x='DANGCEM', y="ZENITHB", data=returns)
plt.show()

### III) Check for Symmetry & Positive Definiteness

In [ ]:
# def is_symmetric(matrix):
#     if np.allclose(matrix, matrix.T):
#         return "Matrix is Symmetric matrix"
#     return "Matrix is a Non-Symmetric matrix"

In [ ]:
# # Checks for Positive Definiteness
# def is_positive_definite(matrix):
#     eigvals = np.linalg.eigvals(matrix) # Eigen Valeus
#     det = np.linalg.det(matrix) # Determinant
#     if np.all(eigvals > 0) and np.all(det > 0):
#         return "Matrix is Positive Definite"
#     elif np.all(eigvals >= 0) and np.all(det >= 0):
#         return "Matrix is Semipositive Definite"
#     return "Matrix is neither Positive nor Semipositive Definite"

In [ ]:
def is_symmetric_positive_definite(matrix):
    eigvals = np.linalg.eigvals(matrix) # Eigen Valeus
    det = np.linalg.det(matrix) # Determinant
    if np.allclose(matrix, matrix.T):
        if np.all(eigvals > 0) and np.all(det > 0):
            return "Matrix is a Symmetric Positive Definite Matrix"
        elif np.all(eigvals >= 0) and np.all(det >= 0):
            return "Matrix is a Symmetric Semipositive Definite"
    return "Matrix is neither Positive nor Semipositive Definite"    

In [ ]:
is_symmetric_positive_definite(covariance)

### IV) Testing for Skewness

In [ ]:
returns.hist(bins=10);

#### i) To determine how many data points we have on either side of the mean

The below code takes the count of data points greater than the mean and divides it by the total number of data points. This will give us the percentage of data points greater than the mean.



In [ ]:
dangcem_skewness = (len(returns[returns.DANGCEM > returns.DANGCEM.mean()])) / (len(returns))
print("DANGCEM Skewness: ", dangcem_skewness)
gtco_skewness = (len(returns[returns.GTCO > returns.GTCO.mean()])) / (len(returns))
print("GTCO Skewness: ", gtco_skewness)
zenithb_skewness = (len(returns[returns.ZENITHB > returns.ZENITHB.mean()])) / (len(returns))
print("ZENITHB Skewness: ", zenithb_skewness)

**Recall:** A value close to 0.5 suggests a roughly symmetrical distribution, while a value significantly greater than 0.5 suggests a negative skew (more returns below the mean).

#### ii) Conducting a Normality Test 
Using `normaltest()`

In [ ]:
dangcem = stats.normaltest(np.array(returns.DANGCEM))
gtco = stats.normaltest(np.array(returns.GTCO))
zenithb = stats.normaltest(np.array(returns.ZENITHB))
print("==============================================================")
print("The Results of Normality")
print("==============================================================")
print("DANGCEM\npvalue = ", dangcem.pvalue, "\nstatistics =", dangcem.statistic)
print("==============================================================")
print("GTCO\npvalue = ", gtco.pvalue, "\nstatistics =", gtco.statistic)
print("==============================================================")
print("ZENITHB\npvalue = ", zenithb.pvalue, "\nstatistics =", zenithb.statistic)
print("==============================================================")

**Recall:**
- Statistic: A test statistic that measures the deviation from normality. Higher values indicate a greater deviation.
- p-value: The probability of observing the data if it were truly normally distributed. A small p-value (typically less than 0.05) suggests that the data is not normally distributed.

#### iii) Testing Skewness and Kurtosis 
Using `stats.jarque_bera()`
##### If **p_value < 0.05** indicates that the data is **not** normally distributed

In [ ]:
dangcem_pvalue = stats.jarque_bera(np.array(returns.DANGCEM)).pvalue
gtco_pvalue = stats.jarque_bera(np.array(returns.GTCO)).pvalue
zenithb_pvalue = stats.jarque_bera(np.array(returns.ZENITHB)).pvalue
print("==============================================================")
print("P-values of Each Equity using Jarque_bera Normality Test")
print("==============================================================")
print("DANGCEM p_value:", dangcem_pvalue)
print("GTCO p_value:", gtco_pvalue)
print("ZENITHB p_value:", zenithb_pvalue)
print("==============================================================")

### iv) Does Our Gaussian Distribution Break Down?

In [ ]:
# returns_max = returns.max()
# returns_min = returns.min()

#  Print maximum and minimum daily log returns
# num_dev_max = (returns_max - returns.mean())/returns.std()
# num_dev_min = (returns_min - returns.mean())/returns.std()

# Print maximum and minimum daily log returns
print("=================================================================================")
print("1. DANGCEM SAMPLE DATA")

return_max = returns["DANGCEM"].max()
return_min = returns["DANGCEM"].min()

num_dev_max = (return_max - returns["DANGCEM"].mean())/returns["DANGCEM"].std()
num_dev_min= (return_min - returns["DANGCEM"].mean())/returns["DANGCEM"].std()
print("=================================================================================")
print("Maximum return of DANGCEM sample data is: ", round(return_max, 5))
print("Minimum return of DANGCEM sample data is: ", round(return_min, 5))
print("=================================================================================")

# Print num_dev_max and num_dev_min
print("Number of standard deviations from the mean for the maximum return: ", round(num_dev_max, 5))
print("Number of standard deviations from the mean for the minimum return: ", round(num_dev_min, 5))
print("=================================================================================\n")


print("2. GTCO SAMPLE DATA")

return_max = returns["GTCO"].max()
return_min = returns["GTCO"].min()

num_dev_max = (return_max - returns["GTCO"].mean())/returns["GTCO"].std()
num_dev_min= (return_min - returns["GTCO"].mean())/returns["GTCO"].std()
print("=================================================================================")
print("Maximum return of GTCO sample data is: ", round(return_max, 5))
print("Minimum return of GTCO sample data is: ", round(return_min, 5))
print("=================================================================================")

# Print num_dev_max and num_dev_min
print("Number of standard deviations from the mean for the maximum return: ", round(num_dev_max, 5))
print("Number of standard deviations from the mean for the minimum return: ", round(num_dev_min, 5))
print("=================================================================================\n")

print("3. ZENITHB SAMPLE DATA")

return_max = returns["ZENITHB"].max()
return_min = returns["ZENITHB"].min()

num_dev_max = (return_max - returns["ZENITHB"].mean())/returns["ZENITHB"].std()
num_dev_min= (return_min - returns["ZENITHB"].mean())/returns["ZENITHB"].std()
print("=================================================================================")
print("Maximum return of ZENITHB sample data is: ", round(return_max, 5))
print("Minimum return of ZENITHB sample data is: ", round(return_min, 5))
print("=================================================================================")

# Print num_dev_max and num_dev_min
print("Number of standard deviations from the mean for the maximum return: ", round(num_dev_max, 5))
print("Number of standard deviations from the mean for the minimum return: ", round(num_dev_min, 5))
print("=================================================================================\n")

In [ ]:
stats.norm.cdf(-0.105361)

In [ ]:
# desired_order = [
#     'AnnualizedReturn',
#     'DailyReturn',
#     'StandardDeviation',
#     'MovingAverageVolatility',
#     'SharpeRatio',
#     'SemiVariance',
#     'HighMinusLow'
# ]

# summary_df = summary_df[desired_order].copy()

# percent_cols = ['AnnualizedReturn', 'DailyReturn', 'StandardDeviation', 
#                 'MovingAverageVolatility', 'HighMinusLow']

# summary_df[percent_cols] = summary_df[percent_cols] * 100

# summary_df = summary_df.style.format({
#     'AnnualizedReturn': "{:.2f}%",
#     'DailyReturn': "{:.3f}%",
#     'StandardDeviation': "{:.2f}%",
#     'MovingAverageVolatility': "{:.2f}%",
#     'HighMinusLow': "{:.2f}%",
#     'SharpeRatio': "{:.4f}",
#     'SemiVariance': "{:.6f}"
# }).set_caption("Summary Statistics of the Three Equities")

In [ ]:
desired_order = [
    'AnnualizedReturn',
    'DailyReturn',
    'StandardDeviation',
    'MovingAverageVolatility',
    'SharpeRatio',
    'SemiVariance',
    'HighMinusLow'
]

# Work with the DataFrame first
summary_df = summary_df[desired_order].copy()

# Convert selected columns to percentages (numeric * 100)
percent_cols = ['AnnualizedReturn', 'DailyReturn', 'StandardDeviation', 
                'MovingAverageVolatility', 'HighMinusLow']

summary_df[percent_cols] = summary_df[percent_cols] * 100

# Only after all numeric work, apply styling
summary_df = summary_df.style.format({
    'AnnualizedReturn': "{:.2f}%",
    'DailyReturn': "{:.3f}%",
    'StandardDeviation': "{:.2f}%",
    'MovingAverageVolatility': "{:.2f}%",
    'HighMinusLow': "{:.2f}%",
    'SharpeRatio': "{:.4f}",
    'SemiVariance': "{:.6f}"
}).set_caption("Summary Statistics of the Three Equities")

In [ ]:
summary_df

In [ ]:
# After creating your styled_summary
summary_df.to_excel("../outputs/summary_statistics.xlsx", engine="openpyxl")

**Interpretation of Summary Statistics**

The table above presents key risk and return metrics for the three stocks over the observed period of 5 years:

- DANGCEM stands out as the strongest performer on a risk-adjusted basis. It achieved the highest daily return (0.001248) while maintaining the lowest volatility (Standard Deviation = 0.021055). This is further confirmed by its highest Sharpe Ratio (0.0593), indicating superior return per unit of risk.
- ZENITHBANK delivered a very similar daily return to DANGCEM (0.001243) but with noticeably higher volatility (0.022772) and a lower Sharpe Ratio (0.0546). This suggests it carried more risk for roughly the same return.
- GTCO recorded the lowest daily return (0.001135) and relatively moderate volatility. It appears to be the most conservative of the three but also the least rewarding in terms of return.


**Overall Insights:**

1. DANGCEM is the clear winner in terms of risk-adjusted performance among the three stocks.
2. Banking stocks (ZENITHBANK and GTCO) showed higher volatility compared to the Industrial Goods stock (DANGCEM), which is expected due to their sensitivity to interest rates and regulatory changes.
3. The combination of DANGCEM with either of the banks could offer good diversification, especially considering the low similarity measures observed earlier.

### Similarity Measures

#### DANGCEM vs ZENITHB

In [ ]:
stock_A = returns.DANGCEM
stock_B = returns.ZENITHB

# Euclidean Distance
euclidean_distance = np.sqrt(np.sum((stock_A - stock_B)**2))

# Manhattan Distance
manhattan_distance = np.sum(np.abs(stock_A-stock_B))

# Euclidean Distance Cosine Similarity
dot_product = np.dot(stock_A, stock_B)
magnitude_A = np.linalg.norm(stock_A)
magnitude_B = np.linalg.norm(stock_B)
cosine_similarity = dot_product / (magnitude_A * magnitude_B)

DANGCEM_ZENITH_similarity_measures = pd.DataFrame(
    {
        "Euclidean Distance": euclidean_distance,
        "Manhattan Distance": manhattan_distance,
        "Cosine Similarity": cosine_similarity
    }, index=["DANGCEM - ZENITHBANK"]
)

#### DANGCEM vs GTCO

In [ ]:
stock_A = returns.DANGCEM
stock_B = returns.GTCO

# Euclidean Distance
euclidean_distance = np.sqrt(np.sum((stock_A - stock_B)**2))

# Manhattan Distance
manhattan_distance = np.sum(np.abs(stock_A-stock_B))

# Euclidean Distance Cosine Similarity
dot_product = np.dot(stock_A, stock_B)
magnitude_A = np.linalg.norm(stock_A)
magnitude_B = np.linalg.norm(stock_B)
cosine_similarity = dot_product / (magnitude_A * magnitude_B)

DANGCEM_GTCO_similarity_measures = pd.DataFrame(
    {
        "Euclidean Distance": euclidean_distance,
        "Manhattan Distance": manhattan_distance,
        "Cosine Similarity": cosine_similarity
    }, index=["DANGCEM - GTCO"]
)

#### ZENITHBANK vs GTCO

In [ ]:
stock_A = returns.ZENITHB
stock_B = returns.GTCO

# Euclidean Distance
euclidean_distance = np.sqrt(np.sum((stock_A - stock_B)**2))

# Manhattan Distance
manhattan_distance = np.sum(np.abs(stock_A-stock_B))

# Euclidean Distance Cosine Similarity
dot_product = np.dot(stock_A, stock_B)
magnitude_A = np.linalg.norm(stock_A)
magnitude_B = np.linalg.norm(stock_B)
cosine_similarity = dot_product / (magnitude_A * magnitude_B)

ZENITHBANK_GTCO_similarity_measures = pd.DataFrame(
    {
        "Euclidean Distance": euclidean_distance,
        "Manhattan Distance": manhattan_distance,
        "Cosine Similarity": cosine_similarity
    }, index=["ZENITHBANK - GTCO"]
)

In [ ]:
# similarity_measures
similarity_measures = pd.concat(
    [
        DANGCEM_ZENITH_similarity_measures,
        DANGCEM_GTCO_similarity_measures,
        ZENITHBANK_GTCO_similarity_measures
    ],
    axis=0
)

In [ ]:
similarity_measures.to_excel("../outputs/similarity_measures.xlsx", engine="openpyxl")

similarity_measures

**Pairwise Similarity Analysis**
> The analysis reveals that ZENITHBANK and GTCO exhibit moderate similarity in their return patterns (Cosine Similarity = 0.563), while DANGCEM shows low similarity with both banking stocks. This suggests good diversification potential when combining DANGCEM with banking stocks.

**Key Insights:**

- ZENITHBANK and GTCO (both from the Financial Services sector) show the highest similarity in their return behavior. This makes sense, banks tend to react similarly to interest rate changes, economic policy, and regulatory news.
- DANGCEM (Industrial Goods / Cement sector) has very low similarity with both banks. This is excellent for diversification, DANGCEM moves differently from the banking stocks.
- Overall, the three stocks are not highly correlated in their daily movements, which is positive from a portfolio diversification perspective.